# 🎯 DipRadar — Bootstrap ML (Colab)

Corre cada célula **uma de cada vez**, por ordem.
O Parquet acumula no Google Drive — podes fechar o Colab entre sessões sem perder progresso.

> ⚠️ **Cada sessão nova do Colab**: volta a correr as células 1, 2, 3 e 4 antes de continuar.

---

## Arquitectura do Pipeline

```
bootstrap_ml.py --layer price   →  ml_training_price.parquet  (Camada A: preço + macro, 20 anos)
bootstrap_ml.py --layer fund    →  ml_training_fund.parquet   (Camada B: fundamentais PIT, 3 anos, US only)
bootstrap_ml.py --skip-backfill →  merge + train_model.py     (merge interno + treino ensemble)
```

**O merge e o treino são feitos pelo `bootstrap_ml.py` — não há código Python de negócio neste notebook.**

---

## Plano de execução

**Fase A — Camada A (Price + Macro, 20 anos):**

| Célula | Slice | Tempo estimado |
|--------|-------|----------------|
| 5 | 0–200 | ~25 min |
| 6 | 200–400 | ~25 min |
| 7 | 400–600 | ~25 min |
| 8 | 600–800 | ~25 min |
| 9 | 800–fim | ~10 min |

**Fase B — Camada B (Fundamentais PIT, YEARS_FUND=3, US only):**

| Célula | Slice | Notas |
|--------|-------|-------|
| 10 | 0–200 | `--force-full` (limpa Parquet antigo) |
| 11 | 200–400 | sem `--force-full` |
| 12 | 400–600 | sem `--force-full` |
| 13 | 600–fim | sem `--force-full` — tickers EU (.DE .PA .L etc.) são saltados automaticamente pelo EU_SUFFIXES filter |

**Fase C — Merge + Treino:**

| Célula | O que faz |
|--------|-----------|
| 14 | `bootstrap_ml.py --skip-backfill` — merge interno + treino ensemble |
| 15 | Validação do contrato de features (anti Training-Serving Skew) |
| 16 | Verificação dos ficheiros gerados |
| 17 | Deploy para Railway |

> 🛡️ **Point-in-Time rigoroso**: reporting lag de 45 dias + fallback NaN. `--force-full` só no primeiro batch da Fase B — apaga dados do Parquet antigo para garantir janela limpa de 3 anos.

## 1 · Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

## 2 · Instalar dependências

In [ ]:
%pip install -q yfinance scikit-learn pyarrow fastparquet xgboost lightgbm shap pandas-datareader
print('✅ Dependências instaladas')

## 3 · Clonar / actualizar repositório

In [ ]:
import os

REPO_DIR  = '/content/DipRadar'
DRIVE_DIR = '/content/drive/MyDrive/DipRadar'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull --quiet
    print('✅ Repositório actualizado')
else:
    !git clone --quiet https://github.com/romeurf/DipRadar.git $REPO_DIR
    %cd $REPO_DIR
    print('✅ Repositório clonado')

os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'📁 Drive dir: {DRIVE_DIR}')

## 4 · (Opcional) Configurar API Key (Tiingo)

> O `bootstrap_ml.py` usa exclusivamente `yfinance` — esta célula é opcional e só necessária para outros scripts do projecto que usem Tiingo.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['TIINGO_API_KEY'] = userdata.get('TIINGO_API_KEY')
    print('✅ TIINGO_API_KEY carregada dos Secrets do Colab')
except Exception:
    print('ℹ️  TIINGO_API_KEY não configurada — não é necessária para bootstrap_ml.py')

---
## ⚡ FASE A — Camada A: Preços + Macro (20 anos)
---

## 5 · Batch 1 — tickers 0–200

> O `build_global_macro_df()` corre automaticamente no início deste batch (3 pedidos de rede: VIX, SPY, T10Y2Y).  
> Se a sessão expirar a meio, volta a correr — os tickers já processados são saltados.

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 6 · Batch 2 — tickers 200–400

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 7 · Batch 3 — tickers 400–600

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 8 · Batch 4 — tickers 600–800

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 600 800 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 9 · Batch 5 — tickers 800 até ao fim

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 800 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

---
## 🧮 FASE B — Camada B: Fundamentais Point-in-Time (YEARS_FUND=3, US only)

> Enriquece cada alerta do Parquet com `gross_margin`, `fcf_yield`, `revenue_growth` e `de_ratio`  
> extraídos dos quarterly statements **já públicos na data do dip** (reporting lag 45 dias).  
> Fallback é **NaN** — nunca `tk.info`.
>
> **`EU_SUFFIXES`**: tickers com `.DE`, `.PA`, `.L`, `.AS`, `.SW`, `.MC`, `.MI`, `.ST`, `.CO`, `.OL`, `.HE`, `.BR`, `.I`, `.LS`, `.VI`, `.WA`  
> são saltados instantaneamente em **todos os batches** — o yfinance não tem fundamentais históricos gratuitos para esses mercados.
>
> **`--force-full` apenas no Batch 1** (célula 10) — apaga o Parquet fund anterior para garantir janela limpa de 3 anos.  
> Nos batches seguintes omite-se para não apagar o trabalho acumulado.
---

## 10 · Fund Batch 1 — tickers 0–200

> `--force-full` limpa o Parquet fund antigo. Só usar neste primeiro batch.

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --force-full \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 11 · Fund Batch 2 — tickers 200–400

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 12 · Fund Batch 3 — tickers 400–600

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 13 · Fund Batch 4 — tickers 600–fim

> Os tickers europeus presentes neste slice são saltados automaticamente pelo `EU_SUFFIXES` filter — sem erro, sem dados corrompidos.

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 600 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

---
## 🏋️ FASE C — Merge + Treino do Ensemble
---

## 14 · Merge Camada A + Camada B + Treino robusto (Tier A+B+C)

> Só depois de todos os batches (Fase A + Fase B) estarem completos.
> O `bootstrap_ml.py --skip-backfill` agora delega o treino da **Camada B** ao novo `train_model.train_all` (Tier A+B+C):
>
>   1. Lê `ml_training_price.parquet` + `ml_training_fund.parquet`
>   2. Faz merge dedupe `(symbol, alert_date)` preferindo **fundamentais reais** sobre fallback
>   3. Grava `ml_training_merged.parquet`
>   4. Chama `train_model.train_all(...)` que faz:
>      - Split cronológico T1/T2/T3 por data (default `train<2021-01`, `cal<2023-01`, `test≥2023-01`)
>      - TimeSeriesSplit CV (sem leakage) + sample-weights por recência (`half_life=1.5y`)
>      - Ensemble RF + XGBoost + LightGBM com voto soft + calibração isotónica em T2
>      - Threshold por regime VIX (low/medium/high) com salvaguarda contra recall<5%
>   5. Grava `dip_model_stage1.pkl`, `dip_model_stage2.pkl`, `ml_report.json` enriquecido
>
> `--exclude-years 2020` é **opcional** (o decay exponencial por recência já reduz 2020 a ~6% do peso de 2024). Mantemos a flag para quem quiser excluir explicitamente o regime COVID.

In [ ]:
!python bootstrap_ml.py \
    --skip-backfill \
    --exclude-years 2020 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 15 · Validar contrato de features (anti Training-Serving Skew)

> Confirma que os `.pkl` treinados e o `ml_features.py` em produção têm **exactamente as mesmas features**, pela mesma ordem.  
> **Não saltes este passo** — é o guarda que impede um crash silencioso em produção.

In [ ]:
import joblib, pathlib
from ml_features import FEATURE_COLUMNS, N_FEATURES

drive_dir = pathlib.Path('/content/drive/MyDrive/DipRadar')

for pkl_name in ['dip_model_stage1.pkl', 'dip_model_stage2.pkl']:
    pkl_path = drive_dir / pkl_name
    if not pkl_path.exists():
        print(f'⚠  {pkl_name} não encontrado — corre a Fase C primeiro')
        continue

    bundle = joblib.load(pkl_path)
    # Both v1 (`feature_columns`) and v2 (Tier A+B+C also writes `feature_columns`)
    # use the same key. The legacy `feature_names` alias is only used internally
    # by sklearn-style estimators.
    model_features = list(bundle.get('feature_columns') or bundle.get('feature_names') or [])
    code_features  = list(FEATURE_COLUMNS)

    if model_features == code_features:
        print(f'✅{pkl_name} — contrato OK ({N_FEATURES} features)')
    else:
        missing = set(code_features) - set(model_features)
        extra   = set(model_features) - set(code_features)
        print(f'❌{pkl_name} — MISMATCH (Training-Serving Skew)!')
        if missing: print(f'  Faltam no modelo : {missing}')
        if extra:   print(f'  Extra no modelo  : {extra}')
        raise AssertionError(f'{pkl_name}: corrige antes de fazer deploy.')

# Sanity-check da meta-info do modelo Tier A+B+C
for pkl_name in ['dip_model_stage1.pkl', 'dip_model_stage2.pkl']:
    pkl_path = drive_dir / pkl_name
    if not pkl_path.exists():
        continue
    bundle = joblib.load(pkl_path)
    alg = bundle.get('algorithm', '?')
    thr = bundle.get('threshold', '?')
    print(f'   {pkl_name}: alg={alg}  threshold={thr}')

## 16 · Verificar todos os ficheiros gerados

In [ ]:
import pathlib, pandas as pd

drive_dir = pathlib.Path('/content/drive/MyDrive/DipRadar')
print('📁 Ficheiros no Drive:')
for f in sorted(drive_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>8.1f} KB')

merged_pq = drive_dir / 'ml_training_merged.parquet'
if merged_pq.exists():
    df = pd.read_parquet(merged_pq)
    print(f'\n📊 Merged Parquet: {len(df):,} alertas | {df["symbol"].nunique()} tickers únicos')
    print('\nDistribuição outcome_label:')
    print(df['outcome_label'].value_counts().to_string())
    fund_cols = ['gross_margin', 'fcf_yield', 'revenue_growth', 'de_ratio']
    existing  = [c for c in fund_cols if c in df.columns]
    if existing:
        print('\n🧮 Cobertura dos Fundamentais PIT (% não-NaN):')
        for col in existing:
            pct = df[col].notna().mean() * 100
            print(f'  {col:<20} {pct:>5.1f}% preenchido')
else:
    print('⚠️  Merged Parquet ainda não existe — corre a Fase C primeiro.')

## 17 · Deploy do modelo para o Railway

### Opção A — Railway Volume (recomendado)
O Railway tem um Volume montado em `/data`.
Descarrega os `.pkl` do Drive para a tua máquina local e faz upload via Railway CLI:
```bash
railway volume cp ./dip_model_stage1.pkl /data/dip_model_stage1.pkl
railway volume cp ./dip_model_stage2.pkl /data/dip_model_stage2.pkl
```

### Opção B — Commit directo (só se < 50 MB por ficheiro)
```bash
git add dip_model_stage1.pkl dip_model_stage2.pkl
git commit -m 'feat: trained ML models — YEARS_FUND=3, EU_SUFFIXES filter'
git push
```